In [1]:
# ruff: noqa

import contextlib
import dataclasses
import datetime
import faulthandler
import os
import signal
import time
# from moviepy.editor import ImageSequenceClip
import numpy as np

import sys 
sys.path.append("/home/franka_deoxys/openpi/packages/openpi-client/src")


from openpi_client import image_tools
from openpi_client import websocket_client_policy
import pandas as pd
from PIL import Image
# from droid.robot_env import RobotEnv
import tqdm
import tyro

sys.path.append("/home/franka_deoxys/openpi/src")

# from openpi.policies import droid_policy
from openpi.policies import franka_policy

faulthandler.enable()

# DROID data collection frequency -- we slow down execution to match this frequency
DROID_CONTROL_FREQUENCY = 15

/home/franka_deoxys/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@dataclasses.dataclass
class Args:
    # Hardware parameters
    left_camera_id: str = "<your_camera_id>"  # e.g., "24259877"
    right_camera_id: str = "<your_camera_id>"  # e.g., "24514023"
    wrist_camera_id: str = "<your_camera_id>"  # e.g., "13062452"

    # Policy parameters
    external_camera: str | None = (
        None  # which external camera should be fed to the policy, choose from ["left", "right"]
    )

    # Rollout parameters
    max_timesteps: int = 600
    # How many actions to execute from a predicted action chunk before querying policy server again
    # 8 is usually a good default (equals 0.5 seconds of action execution).
    open_loop_horizon: int = 8

    # Remote server parameters
    # remote_host: str = "0.0.0.0"  # point this to the IP address of the policy server, e.g., "192.168.1.100"
    # remote_host: str = "192.168.1.96" 
    # remote_host: str = "132.177.8.84"
    remote_host: str = "132.177.8.86"
    remote_port: int = (
        8000  # point this to the port of the policy server, default server port for openpi servers is 8000
    )

In [3]:
# We are using Ctrl+C to optionally terminate rollouts early -- however, if we press Ctrl+C while the policy server is
# waiting for a new action chunk, it will raise an exception and the server connection dies.
# This context manager temporarily prevents Ctrl+C and delays it after the server call is complete.
@contextlib.contextmanager
def prevent_keyboard_interrupt():
    """Temporarily prevent keyboard interrupts by delaying them until after the protected code."""
    interrupted = False
    original_handler = signal.getsignal(signal.SIGINT)

    def handler(signum, frame):
        nonlocal interrupted
        interrupted = True

    signal.signal(signal.SIGINT, handler)
    try:
        yield
    finally:
        signal.signal(signal.SIGINT, original_handler)
        if interrupted:
            raise KeyboardInterrupt

In [4]:
def _extract_observation(args: Args, obs_dict, *, save_to_disk=False):
    image_observations = obs_dict["image"]
    left_image, right_image, wrist_image = None, None, None
    for key in image_observations:
        # Note the "left" below refers to the left camera in the stereo pair.
        # The model is only trained on left stereo cams, so we only feed those.
        if args.left_camera_id in key and "left" in key:
            left_image = image_observations[key]
        elif args.right_camera_id in key and "left" in key:
            right_image = image_observations[key]
        elif args.wrist_camera_id in key and "left" in key:
            wrist_image = image_observations[key]

    # Drop the alpha dimension
    left_image = left_image[..., :3]
    right_image = right_image[..., :3]
    wrist_image = wrist_image[..., :3]

    # Convert to RGB
    left_image = left_image[..., ::-1]
    right_image = right_image[..., ::-1]
    wrist_image = wrist_image[..., ::-1]

    # In addition to image observations, also capture the proprioceptive state
    robot_state = obs_dict["robot_state"]
    cartesian_position = np.array(robot_state["cartesian_position"])
    joint_position = np.array(robot_state["joint_positions"])
    gripper_position = np.array([robot_state["gripper_position"]])

    # Save the images to disk so that they can be viewed live while the robot is running
    # Create one combined image to make live viewing easy
    if save_to_disk:
        combined_image = np.concatenate([left_image, wrist_image, right_image], axis=1)
        combined_image = Image.fromarray(combined_image)
        combined_image.save("robot_camera_views.png")

    return {
        "left_image": left_image,
        "right_image": right_image,
        "wrist_image": wrist_image,
        "cartesian_position": cartesian_position,
        "joint_position": joint_position,
        "gripper_position": gripper_position,
    }

In [5]:
args=Args()

In [ ]:
# Initialize the Panda environment. Using joint velocity action space and gripper position action space is very important.
# env = RobotEnv(action_space="joint_velocity", gripper_action_space="position")

In [6]:
args.remote_host, args.remote_port

('132.177.8.86', 8000)

In [7]:
# Connect to the policy server
policy_client = websocket_client_policy.WebsocketClientPolicy(args.remote_host, args.remote_port)




In [8]:
# instruction = input("Enter instruction: ")
instruction = "pick up"

# Rollout parameters
actions_from_chunk_completed = 0
pred_action_chunk = None

In [9]:
# request_data = droid_policy.make_droid_example()
request_data = franka_policy.make_franka_example()
request_data['prompt'] = instruction
request_data.keys()

dict_keys(['observation/image', 'observation/wrist_image', 'observation/joint_position', 'observation/gripper_position', 'prompt'])

In [10]:
for key in request_data.keys():
    if key not in ["prompt"]:
        print(f"key={key} shape={request_data[key].shape} dtype={request_data[key].dtype}")

key=observation/image shape=(120, 160, 3) dtype=uint8
key=observation/wrist_image shape=(120, 160, 3) dtype=uint8
key=observation/joint_position shape=(7,) dtype=float64
key=observation/gripper_position shape=(1,) dtype=float64


In [11]:
# Wrap the server call in a context manager to prevent Ctrl+C from interrupting it
# Ctrl+C will be handled after the server call is complete
with prevent_keyboard_interrupt():
    # this returns action chunk [10, 8] of 10 joint velocity actions (7) + gripper position (1)
    pred_action_chunk = policy_client.infer(request_data)["actions"]

In [12]:
pred_action_chunk.shape

(10, 7)

In [13]:
print("0:", pred_action_chunk[0] ) 
print("1:", pred_action_chunk[1] )
print("2:", pred_action_chunk[2] )
print("3:", pred_action_chunk[3] )

0: [ 0.62065216  0.58478847  0.24853367  3.35501195  0.31923351 -2.19587702
 -0.21660912]
1: [ 0.62065216  0.58478847  0.24853367  3.35501195  0.31923351 -2.19587702
 -0.21660912]
2: [ 0.62065216  0.58478847  0.24853367  3.35501195  0.31923351 -2.19587702
 -0.21660912]
3: [ 0.62065216  0.58478847  0.24853367  3.35501195  0.31923351 -2.19587702
 -0.21660912]


In [14]:
actions_from_chunk_completed = 0

### Now, inference on Franka data

In [15]:
import h5py
import matplotlib.pyplot as plt

In [16]:
dataset_path = "/home/franka_deoxys/coffee_pod_sujosh70.hdf5"

h5py_file = h5py.File(dataset_path, 'r')
demos= [demo_name for demo_name in h5py_file['data'].keys()]
len(demos)

70

In [17]:
demo_name="demo_0"
demo= h5py_file['data'][demo_name]
demo.keys()

<KeysViewHDF5 ['actions', 'obs']>

In [18]:
demo['actions'].shape

(189, 7)

In [19]:
for key in  demo['obs'].keys():
    print(f"{key}: {demo['obs'][key].shape} {demo['obs'][key].dtype}")

agentview_rgb: (189, 120, 160, 3) uint8
ee_states: (189, 16) float64
eye_in_hand_rgb: (189, 120, 160, 3) uint8
gripper_states: (189, 1) float64
joint_states: (189, 7) float64


In [ ]:
# key=observation/image shape=(120, 160, 3) dtype=uint8
# key=observation/wrist_image shape=(120, 160, 3) dtype=uint8
# key=observation/joint_position shape=(7,) dtype=float64
# key=observation/gripper_position shape=(1,) dtype=float64

In [20]:
import numpy as np
import cv2

# Sample input from demo['obs']
agentview_rgb = demo['obs']['agentview_rgb'][0]         # shape: (120, 160, 3)
eye_in_hand_rgb = demo['obs']['eye_in_hand_rgb'][0]     # shape: (120, 160, 3)
joint_states = demo['obs']['joint_states'][0]           # shape: (7,)
gripper_states = demo['obs']['gripper_states'][0]       # shape: (1,)

# Resize images to (224, 224, 3) using OpenCV
def resize_image_cv2(img):
    resized = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)
    return resized.astype(np.uint8)

# Construct converted observation dictionary
converted_obs = {
    'observation/image': agentview_rgb,
    'observation/wrist_image': eye_in_hand_rgb,
    'observation/joint_position': joint_states,
    'observation/gripper_position': gripper_states,
}

# Print the result
for k, v in converted_obs.items():
    print(f"key={k} shape={v.shape} dtype={v.dtype}")


key=observation/image shape=(120, 160, 3) dtype=uint8
key=observation/wrist_image shape=(120, 160, 3) dtype=uint8
key=observation/joint_position shape=(7,) dtype=float64
key=observation/gripper_position shape=(1,) dtype=float64


In [21]:
converted_obs['prompt']='pick up the coffee pod'

converted_obs.keys()

dict_keys(['observation/image', 'observation/wrist_image', 'observation/joint_position', 'observation/gripper_position', 'prompt'])

In [22]:
# Wrap the server call in a context manager to prevent Ctrl+C from interrupting it
# Ctrl+C will be handled after the server call is complete
with prevent_keyboard_interrupt():
    # this returns action chunk [10, 8] of 10 joint velocity actions (7) + gripper position (1)
    pred_action_chunk = policy_client.infer(converted_obs)["actions"]

In [23]:
print("0:", pred_action_chunk[0] ) 
print("1:", pred_action_chunk[1] )
print("2:", pred_action_chunk[2] )
print("3:", pred_action_chunk[3] )

0: [ 0.01431591 -0.29004455  0.10377378 -0.17821486 -0.00730313 -0.10420549
 -0.21660912]
1: [ 0.01431591 -0.29004455  0.10377378 -0.17821486 -0.00730313 -0.10420549
 -0.21660912]
2: [ 0.01431591 -0.29004455  0.10377378 -0.17821486 -0.00730313 -0.10420549
 -0.21660912]
3: [ 0.01431591 -0.29004455  0.10377378 -0.17821486 -0.00730313 -0.10420549
 -0.21660912]


In [24]:
pred_action_chunk.shape

(10, 7)